In [1]:
# =============================================================================
# BCT LLM Agent Challenge — Signal Extraction Pipeline
# Run this on Kaggle Notebooks. All three datasets are pre-mounted.
# Output: persona_signals.csv — lightweight file to feed into your LangGraph agent
# =============================================================================

# %% [markdown]
# ## 0. Install & Imports

# %%
# pip install kagglehub[pandas-datasets] — already available on Kaggle
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import json
import ast
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
import kagglehub
path = kagglehub.dataset_download("dongrelaxman/amazon-reviews-dataset")
import os
print("Files in Amazon dataset:")
for f in os.listdir(path):
    print(f" -", f)

Files in Amazon dataset:
 - Amazon_Reviews.csv


In [3]:
path = kagglehub.dataset_download("bahramjannesarr/goodreads-book-datasets-10m")
print("Files in Goodreads dataset:")
for f in os.listdir(path):
    print(f" -", f)

Files in Goodreads dataset:
 - book300k-400k.csv
 - user_rating_6000_to_11000.csv
 - user_rating_3000_to_4000.csv
 - book1500k-1600k.csv
 - book3000k-4000k.csv
 - book200k-300k.csv
 - book1900k-2000k.csv
 - book800k-900k.csv
 - book1800k-1900k.csv
 - user_rating_4000_to_5000.csv
 - user_rating_0_to_1000.csv
 - book900k-1000k.csv
 - book700k-800k.csv
 - user_rating_5000_to_6000.csv
 - book600k-700k.csv
 - book500k-600k.csv
 - book1700k-1800k.csv
 - book1000k-1100k.csv
 - book100k-200k.csv
 - book2000k-3000k.csv
 - user_rating_2000_to_3000.csv
 - book1100k-1200k.csv
 - book400k-500k.csv
 - book1300k-1400k.csv
 - book1600k-1700k.csv
 - book1-100k.csv
 - book1400k-1500k.csv
 - book4000k-5000k.csv
 - user_rating_1000_to_2000.csv
 - book1200k-1300k.csv


In [4]:
# =============================================================================
# %% [markdown]
# ## 1. Load Datasets
# Each dataset needs a specific file_path pointing to the right file inside
# the Kaggle dataset. Run a quick list() first if unsure what files exist.
# =============================================================================

# =============================================================================
# CELL 1 — Load All Three Datasets
# =============================================================================
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── 1A. YELP ─────────────────────────────────────────────────────────────────
# Unchanged — already working perfectly
print("Loading Yelp user data...")
yelp_user = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "yelp-dataset/yelp-dataset",
    "yelp_academic_dataset_user.json",
    pandas_kwargs={"lines": True, "nrows": 200_000}
)
print(f"  ✓ Yelp users: {len(yelp_user):,} rows")

print("Loading Yelp review data...")
yelp_review = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "yelp-dataset/yelp-dataset",
    "yelp_academic_dataset_review.json",
    pandas_kwargs={"lines": True, "nrows": 500_000}
)
print(f"  ✓ Yelp reviews: {len(yelp_review):,} rows")

print("Loading Yelp business data...")
yelp_business = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "yelp-dataset/yelp-dataset",
    "yelp_academic_dataset_business.json",
    pandas_kwargs={"lines": True}
)
print(f"  ✓ Yelp businesses: {len(yelp_business):,} rows")

# ── 1B. AMAZON ───────────────────────────────────────────────────────────────
# Single file confirmed: Amazon_Reviews.csv
print("\nLoading Amazon reviews...")
import kagglehub
path = kagglehub.dataset_download("dongrelaxman/amazon-reviews-dataset")
import os
amazon_path = os.path.join(path, "Amazon_Reviews.csv")

amazon = pd.read_csv(
    amazon_path,
    nrows=300_000,
    engine="python",           # tolerates malformed rows the C engine chokes on
    encoding="utf-8",
    encoding_errors="replace", # replaces unreadable characters instead of crashing
    on_bad_lines="skip"        # silently skips any rows that still can't be parsed
)
print(f"  ✓ Amazon reviews: {len(amazon):,} rows | cols: {list(amazon.columns)}")
# ── 1D. GOODREADS — BOOK METADATA ────────────────────────────────────────────
# The book*.csv files hold book details — title, genre, author.
# We load one chunk (100K books) purely for genre/category mapping.
# This gets joined to ratings so we can extract genre affinity per user.
print("\nLoading Goodreads book metadata...")
goodreads_books = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "bahramjannesarr/goodreads-book-datasets-10m",
    "book1-100k.csv"
)
print(f"  ✓ Goodreads books: {len(goodreads_books):,} rows | cols: {list(goodreads_books.columns)}")

print("\n✓ All datasets loaded. Proceed to Section 2 to inspect columns.")

Loading Yelp user data...
  ✓ Yelp users: 200,000 rows
Loading Yelp review data...
  ✓ Yelp reviews: 500,000 rows
Loading Yelp business data...
  ✓ Yelp businesses: 150,346 rows

Loading Amazon reviews...
  ✓ Amazon reviews: 21,214 rows | cols: ['Reviewer Name', 'Profile Link', 'Country', 'Review Count', 'Review Date', 'Rating', 'Review Title', 'Review Text', 'Date of Experience']

Loading Goodreads book metadata...
  ✓ Goodreads books: 58,292 rows | cols: ['Id', 'Name', 'RatingDist1', 'pagesNumber', 'RatingDist4', 'RatingDistTotal', 'PublishMonth', 'PublishDay', 'Publisher', 'CountsOfReview', 'PublishYear', 'Language', 'Authors', 'Rating', 'RatingDist2', 'RatingDist5', 'ISBN', 'RatingDist3']

✓ All datasets loaded. Proceed to Section 2 to inspect columns.


In [5]:
# =============================================================================
# %% [markdown]
# ## 2. Inspect — Run These to Find Exact Column Names Before Extracting
# =============================================================================

# %%
print("=== YELP USER COLUMNS ===")
print(yelp_user.dtypes)
print(yelp_user.head(2))

# %%
print("=== YELP REVIEW COLUMNS ===")
print(yelp_review.dtypes)
print(yelp_review.head(2))

# %%
print("=== AMAZON COLUMNS ===")
print(amazon.dtypes)
print(amazon.head(2))

# %%
print("=== GOODREADS COLUMNS ===")
print(goodreads_books.dtypes)
print(goodreads_books.head(2))



=== YELP USER COLUMNS ===
user_id                object
name                   object
review_count            int64
yelping_since          object
useful                  int64
funny                   int64
cool                    int64
elite                  object
friends                object
fans                    int64
average_stars         float64
compliment_hot          int64
compliment_more         int64
compliment_profile      int64
compliment_cute         int64
compliment_list         int64
compliment_note         int64
compliment_plain        int64
compliment_cool         int64
compliment_funny        int64
compliment_writer       int64
compliment_photos       int64
dtype: object
                  user_id    name  review_count        yelping_since  useful  \
0  qVc8ODYU5SZjKXVBgXdI7w  Walker           585  2007-01-25 16:47:26    7217   
1  j14WgRoU_-2ZE1aw1dXrJg  Daniel          4333  2009-01-25 04:35:42   43091   

   funny   cool                                            

In [7]:
# =============================================================================
# CELL 3 — Yelp Signal Extraction
# Columns confirmed: user_id, average_stars, review_count, elite, fans,
# useful, funny, cool, yelping_since | review: stars, text, user_id, business_id
# No changes needed — all columns match exactly.
# =============================================================================

def extract_yelp_signals(user_df, review_df, business_df):
    print("Extracting Yelp signals...")
    user_df = user_df.copy()
 
    # elite column is a comma-separated string of years e.g. "2009,2010,2011"
    # elite_ever = True if the string is non-empty and not 'None'
    user_df['elite_ever'] = user_df['elite'].apply(
        lambda x: bool(x) and str(x).strip() not in ['', 'None', 'none', '[]']
    )
    print(f"  Total users: {len(user_df):,} | Elite: {user_df['elite_ever'].sum():,}")
 
    # Per-user review signals from the review table
    review_agg = review_df.groupby('user_id').agg(
        review_count_actual = ('stars', 'count'),
        avg_stars            = ('stars', 'mean'),
        std_stars            = ('stars', 'std'),
        review_len_avg       = ('text', lambda x: x.str.len().mean()),
        pct_5star            = ('stars', lambda x: (x == 5).mean()),
        pct_above4           = ('stars', lambda x: (x >= 4).mean()),
    ).reset_index()
 
    # Category affinity: join review → business to get categories per user
    # Note: price_range lives inside the 'attributes' JSON column — we extract
    # it safely here rather than assuming it's a top-level column
    biz = business_df.copy()
 
    def extract_price_range(attrs):
        try:
            if pd.isna(attrs):
                return None
            if isinstance(attrs, str):
                import json
                attrs = json.loads(attrs.replace("'", '"'))
            if isinstance(attrs, dict):
                val = attrs.get('RestaurantsPriceRange2') or attrs.get('price_range')
                return float(val) if val and str(val) != 'None' else None
        except:
            return None
 
    if 'attributes' in biz.columns:
        biz['price_range'] = biz['attributes'].apply(extract_price_range)
    else:
        biz['price_range'] = None  # column absent entirely — skip gracefully
 
    review_biz = review_df[['user_id','business_id']].merge(
        biz[['business_id','categories','stars','price_range']],
        on='business_id', how='left'
    )
 
    def top_categories(cat_series, n=3):
        from collections import Counter
        all_cats = []
        for cats in cat_series.dropna():
            if isinstance(cats, str):
                all_cats.extend([c.strip() for c in cats.split(',')])
        return str([c for c, _ in Counter(all_cats).most_common(n)]) if all_cats else '[]'
 
    cat_agg = review_biz.groupby('user_id').agg(
        avg_biz_stars   = ('stars', 'mean'),
        avg_price_range = ('price_range', 'mean'),
        top_categories  = ('categories', top_categories)
    ).reset_index()
 
    # Merge user + review signals + category signals
    signals = user_df[[
        'user_id','name','review_count','average_stars',
        'yelping_since','elite_ever','fans'
    ]].merge(review_agg, on='user_id', how='left')
    signals = signals.merge(cat_agg, on='user_id', how='left')
 
    # Derived features
    signals['rating_bias'] = signals['avg_stars'] - 3.7  # global Yelp mean ≈ 3.7
    signals['price_sensitivity'] = signals['avg_price_range'].fillna(2).apply(
        lambda x: 'low' if x < 1.5 else 'high' if x > 2.5 else 'medium'
    )
    signals['tone_proxy'] = signals.apply(
        lambda r: 'blunt'    if r['review_len_avg'] < 200
        else      'detailed' if r['review_len_avg'] > 500
        else      'balanced', axis=1
    )
    # Calibration from zzhang83 report: >50% 5★ = fake review signal → lower confidence
    signals['confidence_cap'] = signals['pct_5star'].apply(
        lambda x: 0.7 if x > 0.5 else 1.0
    )
    signals['source'] = 'yelp'
 
    print(f"  ✓ Yelp signals ready: {len(signals):,} user personas")
    return signals
 
yelp_signals = extract_yelp_signals(yelp_user, yelp_review, yelp_business)
yelp_signals.head(3)

Extracting Yelp signals...
  Total users: 200,000 | Elite: 23,180
  ✓ Yelp signals ready: 200,000 user personas


,user_id,name,review_count,average_stars,yelping_since,elite_ever,fans,review_count_actual,avg_stars,std_stars,...,pct_5star,pct_above4,avg_biz_stars,avg_price_range,top_categories,rating_bias,price_sensitivity,tone_proxy,confidence_cap,source
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,3.91,2007-01-25 16:47:26,True,267,2.0,5.0,0.00000,...,1.0,1.0,4.25,2.0,"['American (Traditional)', 'Restaurants', 'Sea...",1.3,medium,balanced,0.7,yelp
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,3.74,2009-01-25 04:35:42,True,3138,5.0,3.8,0.83666,...,0.2,0.6,3.70,2.0,"['Food', 'Restaurants', 'Arts & Entertainment']",0.1,medium,detailed,1.0,yelp
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,3.32,2008-07-25 10:41:00,True,52,2.0,5.0,0.00000,...,1.0,1.0,4.00,1.0,"['Food', 'Restaurants', 'Bakeries']",1.3,low,detailed,0.7,yelp


In [9]:
# =============================================================================
# CELL 4 — Amazon Signal Extraction
# Confirmed columns:
#   'Profile Link'    → unique user identifier (no user_id column exists)
#   'Rating'          → string: "Rated X out of 5 stars" — needs parsing
#   'Review Count'    → string: "X review" or "X reviews" — needs parsing
#   'Review Text'     → review body
#   'Country'         → user location
#   'Reviewer Name'   → display name
# =============================================================================

def extract_amazon_signals(df):
    print("Extracting Amazon signals...")
    df = df.copy()
 
    # ── Parse Rating: "Rated 3 out of 5 stars" → 3.0 ──────────────────────────
    def parse_rating(r):
        if pd.isna(r):
            return None
        try:
            # extract the first number from the string
            import re
            match = re.search(r'(\d+(?:\.\d+)?)', str(r))
            return float(match.group(1)) if match else None
        except:
            return None
 
    df['stars'] = df['Rating'].apply(parse_rating)
 
    # ── Parse Review Count: "9 reviews" → 9 ──────────────────────────────────
    def parse_review_count(r):
        if pd.isna(r):
            return 1
        try:
            import re
            match = re.search(r'(\d+)', str(r))
            return int(match.group(1)) if match else 1
        except:
            return 1
 
    df['review_count_parsed'] = df['Review Count'].apply(parse_review_count)
 
    # ── Use Profile Link as user_id ──────────────────────────────────────────

    df['user_id'] = df['Profile Link'].where(
    df['Profile Link'].notna(),
    'unknown_' + df.index.astype(str).to_series().values
    )
 
    # Drop rows with no parseable rating
    df = df.dropna(subset=['stars'])
    df = df[df['stars'].between(1, 5)]
    print(f"  Rows after cleaning: {len(df):,}")
 
    # Per-user aggregation
    agg = df.groupby('user_id').agg(
        review_count    = ('stars', 'count'),
        avg_stars       = ('stars', 'mean'),
        std_stars       = ('stars', 'std'),
        review_len_avg  = ('Review Text', lambda x: x.str.len().mean()),
        reviewer_name   = ('Reviewer Name', 'first'),
        country         = ('Country', 'first'),
        max_review_count = ('review_count_parsed', 'max'),
    ).reset_index()
 
    # Derived features
    agg['rating_bias']       = agg['avg_stars'] - 3.7
    agg['price_sensitivity'] = 'medium'  # Amazon doesn't expose price range directly
    agg['tone_proxy']        = agg['review_len_avg'].apply(
        lambda x: 'detailed' if x > 400
        else      'blunt'    if x < 150
        else      'balanced' if not pd.isna(x)
        else      'unknown'
    )
    agg['confidence_cap'] = 1.0
    agg['elite_ever']     = False
    agg['pct_5star']      = df.groupby('user_id')['stars'].apply(
        lambda x: (x == 5).mean()
    ).values[:len(agg)] if len(agg) > 0 else 0.0
    agg['pct_above4']     = df.groupby('user_id')['stars'].apply(
        lambda x: (x >= 4).mean()
    ).values[:len(agg)] if len(agg) > 0 else 0.0
    agg['top_categories'] = '[]'  # Amazon dataset doesn't include product category
    agg['source']         = 'amazon'
 
    print(f"  ✓ Amazon signals ready: {len(agg):,} user personas")
    return agg
 
amazon_signals = extract_amazon_signals(amazon)
amazon_signals.head(3)

Extracting Amazon signals...
  Rows after cleaning: 21,055
  ✓ Amazon signals ready: 21,055 user personas


,user_id,review_count,avg_stars,std_stars,review_len_avg,reviewer_name,country,max_review_count,rating_bias,price_sensitivity,tone_proxy,confidence_cap,elite_ever,pct_5star,pct_above4,top_categories,source
0,/users/46d1ed150000640001000051,1,4.0,NaN,356.0,Kim Fuglsang Kramer,DK,2,0.3,medium,balanced,1.0,False,0.0,1.0,[],amazon
1,/users/474aaec70000640001000a44,1,5.0,NaN,867.0,Mads Dørup,DK,82,1.3,medium,detailed,1.0,False,1.0,1.0,[],amazon
2,/users/47bd4ffe0000640001001044,1,5.0,NaN,197.0,Anders T,DK,1,1.3,medium,balanced,1.0,False,1.0,1.0,[],amazon


In [10]:
# =============================================================================
# CELL 5 — Goodreads Signal Extraction
# Confirmed columns (book metadata, NOT per-user ratings):
#   'Id', 'Name', 'Authors', 'Language', 'Publisher', 'Rating' (avg),
#   'RatingDist1'–'RatingDist5' (format: "1:9896"), 'RatingDistTotal',
#   'CountsOfReview', 'pagesNumber', 'PublishYear'
#
# Since there are no individual user rows in this file, we extract
# GENRE/CATEGORY KNOWLEDGE for Task B recommendations instead of
# user behavioral fingerprints. This tells the agent which genres are
# popular, which authors dominate, and what rating patterns look like.
# =============================================================================

 
def extract_goodreads_signals(books_df):
    print("Extracting Goodreads signals...")
    df = books_df.copy()
 
    # ── Parse RatingDist columns: "1:9896" → 9896 ────────────────────────────
    def parse_dist(val):
        if pd.isna(val):
            return 0
        try:
            return int(str(val).split(':')[1])
        except:
            return 0
 
    def parse_total(val):
        if pd.isna(val):
            return 1
        try:
            return max(int(str(val).split(':')[1]), 1)
        except:
            return 1
 
    for i in range(1, 6):
        df[f'dist_{i}'] = df[f'RatingDist{i}'].apply(parse_dist)
 
    df['total_ratings'] = df['RatingDistTotal'].apply(parse_total)
 
    # Percentages per star level
    for i in range(1, 6):
        df[f'pct_{i}star'] = df[f'dist_{i}'] / df['total_ratings']
 
    # ── Genre proxy from Authors + Language ──────────────────────────────────
    # Goodreads doesn't expose genre directly in this dataset.
    # We use Language as a proxy for content type and Publisher for category.
    df['language_clean'] = df['Language'].fillna('unknown').str.strip().str.lower()
 
    # ── Build genre-level summary rows ───────────────────────────────────────
    # Each row = one "genre persona" — the behavioral fingerprint of a book type.
    # The agent uses these as reference anchors when making book recommendations.
 
    genre_signals = df.groupby('language_clean').agg(
        book_count      = ('Id', 'count'),
        avg_rating      = ('Rating', 'mean'),
        avg_pages       = ('pagesNumber', 'mean'),
        avg_reviews     = ('CountsOfReview', 'mean'),
        pct_5star       = ('pct_5star', 'mean'),
        pct_1star       = ('pct_1star', 'mean'),
        top_authors     = ('Authors', lambda x: x.value_counts().index[0] if len(x) > 0 else 'Unknown'),
    ).reset_index()
 
    # Shape into persona-compatible schema
    genre_signals = genre_signals.rename(columns={
        'language_clean': 'user_id',  # language group acts as the "persona" here
        'avg_rating':     'avg_stars',
    })
    genre_signals['review_count']    = genre_signals['avg_reviews'].astype(int)
    genre_signals['rating_bias']     = genre_signals['avg_stars'] - 3.7
    genre_signals['std_stars']       = 0.5   # not available at book level
    genre_signals['review_len_avg']  = genre_signals['avg_pages'] * 2  # proxy
    genre_signals['price_sensitivity']= 'medium'
    genre_signals['tone_proxy']      = 'detailed'  # book reviewers tend to be verbose
    genre_signals['confidence_cap']  = genre_signals['pct_5star'].apply(
        lambda x: 0.7 if x > 0.5 else 1.0
    )
    genre_signals['elite_ever']      = False
    genre_signals['pct_above4']      = genre_signals['pct_5star'] + (genre_signals['avg_stars'] - 3.0).clip(0) / 2
    genre_signals['top_categories']  = genre_signals['top_authors'].apply(
        lambda x: str([x])
    )
    genre_signals['source']          = 'goodreads'
 
    print(f"  Note: Goodreads file contains book metadata (no per-user rows).")
    print(f"  Extracted {len(genre_signals):,} genre-level behavioral anchors for Task B.")
    return genre_signals
 
goodreads_signals = extract_goodreads_signals(goodreads_books)
goodreads_signals.head(5)

Extracting Goodreads signals...
  Note: Goodreads file contains book metadata (no per-user rows).
  Extracted 37 genre-level behavioral anchors for Task B.


,user_id,book_count,avg_stars,avg_pages,avg_reviews,pct_5star,pct_1star,top_authors,review_count,rating_bias,std_stars,review_len_avg,price_sensitivity,tone_proxy,confidence_cap,elite_ever,pct_above4,top_categories,source
0,afr,1,3.870000,160.000000,33.000000,0.378430,0.049365,Paulo Coelho,33,0.170000,0.5,320.000000,medium,detailed,1.0,False,0.813430,['Paulo Coelho'],goodreads
1,ara,6,3.635000,476.166667,55.833333,0.490440,0.015086,J.K. Rowling,55,-0.065000,0.5,952.333333,medium,detailed,1.0,False,0.807940,['J.K. Rowling'],goodreads
2,cat,1,3.350000,223.000000,6.000000,0.156250,0.044643,Ferran Torrent,6,-0.350000,0.5,446.000000,medium,detailed,1.0,False,0.331250,['Ferran Torrent'],goodreads
3,en-ca,11,3.898182,246.818182,237.727273,0.325337,0.019777,Madeleine L'Engle,237,0.198182,0.5,493.636364,medium,detailed,1.0,False,0.774428,"[""Madeleine L'Engle""]",goodreads
4,en-gb,425,3.919247,313.294118,80.360000,0.332337,0.019841,Noam Chomsky,80,0.219247,0.5,626.588235,medium,detailed,1.0,False,0.791961,['Noam Chomsky'],goodreads


In [12]:
# =============================================================================
# CELL 6 — Combine All Signals → Master Persona Schema
# =============================================================================

import numpy as np
 
CORE_COLS = [
    'user_id', 'source', 'review_count', 'avg_stars', 'rating_bias',
    'std_stars', 'review_len_avg', 'price_sensitivity', 'tone_proxy',
    'confidence_cap', 'elite_ever', 'pct_5star', 'pct_above4', 'top_categories'
]
 
def standardise(df):
    for c in CORE_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df[CORE_COLS].copy()
 
frames = []
for label, df in [
    ('yelp', yelp_signals),
    ('amazon', amazon_signals),
    ('goodreads', goodreads_signals)
]:
    if df is not None and not df.empty:
        frames.append(standardise(df))
        print(f"  {label}: {len(df):,} rows added")
 
persona_schema = pd.concat(frames, ignore_index=True)
 
print(f"\n✓ Master persona schema: {len(persona_schema):,} total rows")
print(persona_schema['source'].value_counts())
persona_schema.head(3)

  yelp: 200,000 rows added
  amazon: 21,055 rows added
  goodreads: 37 rows added

✓ Master persona schema: 221,092 total rows
source
yelp         200000
amazon        21055
goodreads        37
Name: count, dtype: int64


,user_id,source,review_count,avg_stars,rating_bias,std_stars,review_len_avg,price_sensitivity,tone_proxy,confidence_cap,elite_ever,pct_5star,pct_above4,top_categories
0,qVc8ODYU5SZjKXVBgXdI7w,yelp,585,5.0,1.3,0.00000,335.0,medium,balanced,0.7,True,1.0,1.0,"['American (Traditional)', 'Restaurants', 'Sea..."
1,j14WgRoU_-2ZE1aw1dXrJg,yelp,4333,3.8,0.1,0.83666,2279.8,medium,detailed,1.0,True,0.2,0.6,"['Food', 'Restaurants', 'Arts & Entertainment']"
2,2WnXYQFK0hXEoTxPtV2zvg,yelp,665,5.0,1.3,0.00000,919.5,low,detailed,0.7,True,1.0,1.0,"['Food', 'Restaurants', 'Bakeries']"


In [14]:
# =============================================================================
# CELL 7 — Filter & Sample for Agent Use
# =============================================================================

# Remove sparse users (fewer than 3 reviews — too little signal)
filtered = persona_schema[
    ((persona_schema['source'] == 'yelp')      & (persona_schema['review_count'] >= 3)) |
    ((persona_schema['source'] == 'amazon')    & (persona_schema['review_count'] >= 1)) |
    ((persona_schema['source'] == 'goodreads') & (persona_schema['review_count'] >= 1))
].copy()
 
# Prioritise elite users — all of them + random sample of regular users
elite    = filtered[filtered['elite_ever'] == True]
regular  = filtered[filtered['elite_ever'] == False].sample(
    min(5000, len(filtered[filtered['elite_ever'] == False])),
    random_state=42
)
# Keep all Goodreads genre anchors
goodreads_rows = filtered[filtered['source'] == 'goodreads']
 
final_sample = pd.concat([elite, regular, goodreads_rows], ignore_index=True)
final_sample = final_sample.drop_duplicates(subset=['user_id', 'source'])
 
print(f"Final persona sample: {len(final_sample):,} rows")
print(final_sample['source'].value_counts())


Final persona sample: 28,212 rows
source
yelp         27610
amazon         570
goodreads       32
Name: count, dtype: int64


In [15]:
print(final_sample['tone_proxy'].value_counts())
print(final_sample['price_sensitivity'].value_counts())
print(final_sample['rating_bias'].describe())
print(final_sample['elite_ever'].value_counts())

tone_proxy
detailed    16274
balanced     9693
blunt        2245
Name: count, dtype: int64
price_sensitivity
medium    23548
low        3220
high       1444
Name: count, dtype: int64
count    27967.000000
mean         0.227164
std          0.967119
min         -2.700000
25%         -0.200000
50%          0.300000
75%          0.966667
max          1.300000
Name: rating_bias, dtype: float64
elite_ever
True     23180
False     5032
Name: count, dtype: int64


In [16]:
# =============================================================================
# CELL 8 — Export
# =============================================================================

output_path = "/kaggle/working/persona_signals.csv"
final_sample.to_csv(output_path, index=False)

print(f"✓ Saved → {output_path}")
print(f"  Shape: {final_sample.shape}")
print(f"  Columns: {list(final_sample.columns)}")
print(f"\nSample row:")
print(final_sample.iloc[0].to_dict())

✓ Saved → /kaggle/working/persona_signals.csv
  Shape: (28212, 14)
  Columns: ['user_id', 'source', 'review_count', 'avg_stars', 'rating_bias', 'std_stars', 'review_len_avg', 'price_sensitivity', 'tone_proxy', 'confidence_cap', 'elite_ever', 'pct_5star', 'pct_above4', 'top_categories']

Sample row:
{'user_id': 'qVc8ODYU5SZjKXVBgXdI7w', 'source': 'yelp', 'review_count': 585, 'avg_stars': 5.0, 'rating_bias': 1.2999999999999998, 'std_stars': 0.0, 'review_len_avg': 335.0, 'price_sensitivity': 'medium', 'tone_proxy': 'balanced', 'confidence_cap': 0.7, 'elite_ever': True, 'pct_5star': 1.0, 'pct_above4': 1.0, 'top_categories': "['American (Traditional)', 'Restaurants', 'Seafood']"}


In [ ]:
# =============================================================================
# %% [markdown]
# ## 3. Signal Extraction — Yelp
# Priority: elite users (5-7x richer review history, consistent rating patterns)
# Key finding from analysis reports: fake reviews spike at 5★ — real reviews
# cluster 3.5–4★. We use this to calibrate confidence in Task A outputs.
# =============================================================================

# %%
def extract_yelp_signals(user_df, review_df, business_df):
    """
    Extracts behavioral persona signals from Yelp user + review + business data.
    Returns one row per user with their behavioral fingerprint.
    """

    # ── Step 1: Filter for elite users ────────────────────────────────────────
    # Based on hemachandarn analysis: elite users have ~249 avg reviews vs ~39
    # for regular users. Sampling them gives richer behavioral profiles.
    user_df = user_df.copy()

    # elite column is a list of years as string e.g. "2019, 2020"
    # elite_ever = True if they were ever elite
    user_df['elite_ever'] = user_df['elite'].apply(
        lambda x: bool(x) and x not in ['None', '', '[]', 'none']
        if isinstance(x, str) else bool(x)
    )

    # Keep all users but flag elite ones — we'll weight them later
    print(f"  Total users: {len(user_df):,}")
    print(f"  Elite users: {user_df['elite_ever'].sum():,} ({user_df['elite_ever'].mean()*100:.1f}%)")

    # ── Step 2: Per-user review signals ───────────────────────────────────────
    review_agg = review_df.groupby('user_id').agg(
        review_count_actual = ('stars', 'count'),
        avg_stars            = ('stars', 'mean'),
        std_stars            = ('stars', 'std'),        # rating consistency
        review_len_avg       = ('text', lambda x: x.str.len().mean()),
        review_len_std       = ('text', lambda x: x.str.len().std()),
        pct_5star            = ('stars', lambda x: (x == 5).mean()),
        pct_1star            = ('stars', lambda x: (x == 1).mean()),
        pct_above4           = ('stars', lambda x: (x >= 4).mean()),
    ).reset_index()

    # ── Step 3: Category affinity from business ───────────────────────────────
    # Join review → business to get categories per user
    review_biz = review_df[['user_id','business_id']].merge(
        business_df[['business_id','categories','stars','price_range']],
        on='business_id', how='left'
    )

    def top_categories(cat_series, n=3):
        """Flatten category lists and return top n by frequency."""
        all_cats = []
        for cats in cat_series.dropna():
            if isinstance(cats, str):
                all_cats.extend([c.strip() for c in cats.split(',')])
        if not all_cats:
            return []
        from collections import Counter
        return [c for c, _ in Counter(all_cats).most_common(n)]

    cat_agg = review_biz.groupby('user_id').agg(
        avg_biz_stars     = ('stars', 'mean'),
        avg_price_range   = ('price_range', 'mean'),
        top_categories    = ('categories', top_categories)
    ).reset_index()

    # ── Step 4: Merge everything ───────────────────────────────────────────────
    signals = user_df[[
        'user_id', 'name', 'review_count', 'average_stars',
        'yelping_since', 'elite_ever', 'fans', 'useful', 'funny', 'cool'
    ]].merge(review_agg, on='user_id', how='left')
    signals = signals.merge(cat_agg, on='user_id', how='left')

    # ── Step 5: Derive persona-level features ─────────────────────────────────
    # rating_bias: how this user's avg compares to global mean (3.7)
    global_mean = 3.7
    signals['rating_bias'] = signals['avg_stars'] - global_mean

    # price_sensitivity: low/medium/high based on avg price range of visited places
    signals['price_sensitivity'] = pd.cut(
        signals['avg_price_range'].fillna(2),
        bins=[0, 1.5, 2.5, 5],
        labels=['low', 'medium', 'high']
    )

    # tone_proxy: shorter reviews with extreme ratings → blunt; longer → detailed
    signals['tone_proxy'] = signals.apply(
        lambda r: 'blunt' if (r['review_len_avg'] < 200 and r['std_stars'] > 1.5)
                  else 'detailed' if r['review_len_avg'] > 500
                  else 'balanced', axis=1
    )

    # confidence_cap: flag users whose 5★ rate is suspiciously high (fake review insight)
    # From zzhang83: >50% 5★ is a fake review signal → lower confidence in Task A
    signals['confidence_cap'] = signals['pct_5star'].apply(
        lambda x: 0.7 if x > 0.5 else 1.0
    )

    signals['source'] = 'yelp'
    print(f"  Yelp signals extracted: {len(signals):,} user personas")
    return signals

yelp_signals = extract_yelp_signals(yelp_user, yelp_review, yelp_business)
yelp_signals.head(3)


In [ ]:
# =============================================================================
# %% [markdown]
# ## 4. Signal Extraction — Amazon
# Focus: product category affinity, rating bias, review verbosity, intent signals
# =============================================================================

# %%
def extract_amazon_signals(df):
    """
    Extracts behavioral signals from Amazon review data.
    Column names may vary — update based on your inspect output above.
    Common columns: reviewerID, overall, reviewText, verified, asin, category
    """
    df = df.copy()

    # Normalise column names to lowercase
    df.columns = df.columns.str.lower().str.replace(' ', '_')

    # Map common Amazon column name variants
    col_map = {
        'reviewerid': 'user_id',
        'reviewer_id': 'user_id',
        'overall': 'stars',
        'rating': 'stars',
        'reviewtext': 'review_text',
        'review_text': 'review_text',
        'reviewbody': 'review_text',
        'verified': 'verified_purchase',
        'verifiedpurchase': 'verified_purchase',
    }
    df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})

    required = ['user_id', 'stars']
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"  ⚠ Missing columns after mapping: {missing}")
        print(f"  Available: {list(df.columns)}")
        print("  Update col_map above to match your dataset's actual column names.")
        return pd.DataFrame()

    # Per-user aggregation
    agg = df.groupby('user_id').agg(
        review_count   = ('stars', 'count'),
        avg_stars      = ('stars', 'mean'),
        std_stars      = ('stars', 'std'),
        pct_verified   = ('verified_purchase', 'mean') if 'verified_purchase' in df.columns else ('stars', lambda x: np.nan),
        review_len_avg = ('review_text', lambda x: x.str.len().mean()) if 'review_text' in df.columns else ('stars', lambda x: np.nan),
    ).reset_index()

    # Category affinity
    if 'category' in df.columns or 'asin' in df.columns:
        cat_col = 'category' if 'category' in df.columns else 'asin'
        cat_agg = df.groupby('user_id')[cat_col].apply(
            lambda x: x.value_counts().index[0] if len(x) > 0 else 'unknown'
        ).reset_index(name='top_category')
        agg = agg.merge(cat_agg, on='user_id', how='left')

    agg['rating_bias'] = agg['avg_stars'] - 3.7
    agg['price_sensitivity'] = 'medium'   # Amazon doesn't expose price range directly
    agg['tone_proxy'] = agg['review_len_avg'].apply(
        lambda x: 'detailed' if x > 400 else 'blunt' if x < 150 else 'balanced'
        if not pd.isna(x) else 'unknown'
    )
    agg['confidence_cap'] = 1.0
    agg['elite_ever'] = False
    agg['source'] = 'amazon'

    print(f"  Amazon signals extracted: {len(agg):,} user personas")
    return agg

amazon_signals = extract_amazon_signals(amazon)
if not amazon_signals.empty:
    amazon_signals.head(3)

In [ ]:
# =============================================================================
# %% [markdown]
# ## 5. Signal Extraction — Goodreads
# Focus: genre taste, reading velocity, review verbosity
# =============================================================================

# %%
def extract_goodreads_signals(df):
    """
    Extracts behavioral signals from Goodreads data.
    Common columns: user_id, book_id, rating, review_text, shelf, date_added
    Column names vary by Goodreads dataset variant — inspect first.
    """
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(' ', '_')

    col_map = {
        'userid': 'user_id',
        'bookid': 'book_id',
        'rating': 'stars',
        'score': 'stars',
        'review': 'review_text',
        'shelves': 'shelf',
    }
    df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})

    if 'user_id' not in df.columns or 'stars' not in df.columns:
        print(f"  ⚠ Expected 'user_id' and 'stars'. Found: {list(df.columns)}")
        print("  Update col_map above to match your actual column names.")
        return pd.DataFrame()

    agg = df.groupby('user_id').agg(
        review_count   = ('stars', 'count'),
        avg_stars      = ('stars', 'mean'),
        std_stars      = ('stars', 'std'),
        review_len_avg = ('review_text', lambda x: x.str.len().mean()) if 'review_text' in df.columns else ('stars', lambda x: np.nan),
    ).reset_index()

    # Genre affinity from shelf names
    if 'shelf' in df.columns:
        def top_shelf(shelves, n=3):
            from collections import Counter
            all_s = []
            for s in shelves.dropna():
                if isinstance(s, str):
                    all_s.extend(s.split(','))
            return [x.strip() for x, _ in Counter(all_s).most_common(n)]

        shelf_agg = df.groupby('user_id')['shelf'].apply(top_shelf).reset_index(name='top_genres')
        agg = agg.merge(shelf_agg, on='user_id', how='left')

    agg['rating_bias'] = agg['avg_stars'] - 3.7
    agg['price_sensitivity'] = 'medium'
    agg['tone_proxy'] = agg['review_len_avg'].apply(
        lambda x: 'detailed' if x > 400 else 'blunt' if x < 100 else 'balanced'
        if not pd.isna(x) else 'unknown'
    )
    agg['confidence_cap'] = 1.0
    agg['elite_ever'] = False
    agg['source'] = 'goodreads'

    print(f"  Goodreads signals extracted: {len(agg):,} user personas")
    return agg

goodreads_signals = extract_goodreads_signals(goodreads)
if not goodreads_signals.empty:
    goodreads_signals.head(3)


In [ ]:

# =============================================================================
# %% [markdown]
# ## 6. Combine All Signals → Master Persona Schema
# Standardise columns across all three sources and stack into one file.
# This CSV is your agent's ground truth — lightweight, portable, no raw data.
# =============================================================================

# %%
CORE_COLS = [
    'user_id', 'source', 'review_count', 'avg_stars', 'rating_bias',
    'std_stars', 'review_len_avg', 'price_sensitivity', 'tone_proxy',
    'confidence_cap', 'elite_ever', 'pct_5star', 'pct_above4'
]

def standardise(df, extra_cols=None):
    """Keep only core columns + any source-specific extras. Fill missing."""
    cols = CORE_COLS + (extra_cols or [])
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df[cols]

frames = []
if not yelp_signals.empty:
    frames.append(standardise(yelp_signals, extra_cols=['top_categories']))
if not amazon_signals.empty:
    frames.append(standardise(amazon_signals, extra_cols=['top_category']))
if not goodreads_signals.empty:
    frames.append(standardise(goodreads_signals, extra_cols=['top_genres']))

persona_schema = pd.concat(frames, ignore_index=True)

print(f"\n✓ Master persona schema: {len(persona_schema):,} users across {persona_schema['source'].nunique()} datasets")
print(persona_schema['source'].value_counts())
print("\nSchema preview:")
persona_schema.head(5)

In [ ]:
# =============================================================================
# %% [markdown]
# ## 7. Sample for Agent Use
# The full schema is large. For prompt injection we want a curated sample:
# - Prioritise elite users (richer fingerprints)
# - Ensure coverage across all three sources
# - Remove users with <5 reviews (too sparse for reliable signals)
# =============================================================================

# %%
# Filter out sparse users
filtered = persona_schema[persona_schema['review_count'] >= 5].copy()

# Priority sample: all elite + random from the rest
elite_sample = filtered[filtered['elite_ever'] == True]
regular_sample = filtered[filtered['elite_ever'] == False].sample(
    min(5000, len(filtered[filtered['elite_ever'] == False])),
    random_state=42
)

final_sample = pd.concat([elite_sample, regular_sample], ignore_index=True)

print(f"Final persona sample: {len(final_sample):,} users")
print(f"  Elite: {(final_sample['elite_ever']==True).sum():,}")
print(f"  Regular: {(final_sample['elite_ever']==False).sum():,}")
print(f"  Source breakdown:\n{final_sample['source'].value_counts()}")



In [ ]:
# =============================================================================
# %% [markdown]
# ## 8. Export
# Save as CSV. On Kaggle this goes to /kaggle/working/ — download it from there
# and use it as input in your LangGraph agent containers.
# =============================================================================

# %%
output_path = "/kaggle/working/persona_signals.csv"
final_sample.to_csv(output_path, index=False)
print(f"✓ Saved to {output_path}")
print(f"  File size: ~{final_sample.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory")
print("\nColumn summary:")
print(final_sample.describe(include='all').T[['count','mean','top','freq']].dropna(how='all'))